In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [2]:
from torchvision.transforms import transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trainset = CIFAR10(root="/content/data",train=True,download=True,transform=transform)
testset = CIFAR10(root="/content/data",train=False,download=True,transform=transform)

100%|██████████| 170M/170M [19:55<00:00, 143kB/s]


In [3]:
trainloader = DataLoader(trainset,batch_size=64,shuffle=True)
testloader = DataLoader(testset,batch_size=64)

In [4]:
from torch.nn.modules.activation import ReLU
class CNN(nn.Module):
  def __init__(self):
    super(CNN,self).__init__()

    self.conv_layers = nn.Sequential(
        nn.Conv2d(3,32,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2), #kernelsize & stride

        nn.Conv2d(32,64,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2), #kernelsize & stride

        nn.Conv2d(64,128,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2), #kernelsize & stride
    )


    self.fc_layers = nn.Sequential(
        nn.Linear(4*4*128,256),
        nn.ReLU(),

        nn.Linear(256,10)
    )

  def forward(self,x):
    x = self.conv_layers(x)
    x = x.view(x.size(0),-1) #flattening
    x = self.fc_layers(x)

    return x

In [5]:
model = CNN()
criterion = nn.CrossEntropyLoss()
optimiser = optim.Adam(model.parameters())

In [6]:
epochs =10

for epoch in range(epochs):
  epoch_training_loss = 0.0

  for images,label in trainloader:
    optimiser.zero_grad()

    outputs = model.forward(images)
    loss = criterion(outputs,label)
    loss.backward()
    optimiser.step()

    epoch_training_loss+=loss

  print(f"for epoch {epoch+1}/{epochs} training loss is = {epoch_training_loss/len(trainloader)}")



for epoch 1/10 training loss is = 1.3811231851577759
for epoch 2/10 training loss is = 0.9493564367294312
for epoch 3/10 training loss is = 0.7506524324417114
for epoch 4/10 training loss is = 0.6225475668907166
for epoch 5/10 training loss is = 0.5158325433731079
for epoch 6/10 training loss is = 0.415555864572525
for epoch 7/10 training loss is = 0.33067262172698975
for epoch 8/10 training loss is = 0.255161851644516
for epoch 9/10 training loss is = 0.19780322909355164
for epoch 10/10 training loss is = 0.14938704669475555


In [7]:
correct = 0
total = 0

model.eval()
with torch.no_grad():
  for image,label in testloader:
    outputs = model.forward(image)
    _,predicted = torch.max(outputs,1)

    correct+=(predicted==label).sum().item()
    total+=label.size(0)

print("accuracy=" ,correct/total*100)

accuracy= 74.92999999999999
